# Results for model: tiiuae/falcon3-7b-instruct

In [ ]:
import polars as pl
import xgboost as xgb
from datetime import datetime, timedelta

# Set top quantile
TOP_QUANTILE = 0.15

# Load data
data = pl.read_parquet('./jane_street_data/train.parquet')

# Function to calculate rolling quantile
def rolling_quantile(data, column, window, quantile):
    quantile_rolling = data.rolling(window=window).quantile(column, quantile=quantile, overwrite=True)
    return data.join(quantile_rolling, on='date_id', how='left').set_index('date_id').fill_missing(method='bfill')['quantile']

# Define window size
window_size = 30  # Adjust based on problem specifics

# Calculate rolling quantile for 'feature_00' and 'responder_6'
data = data.with_columns([
    rolling_quantile(data, 'feature_00', window_size, TOP_QUANTILE).alias('feature_00_rolling_top_quantile'),
    rolling_quantile(data, 'responder_6', window_size, TOP_QUANTILE).alias('responder_6_rolling_top_quantile')
])

# Prepare data for XGBoost (label encode if necessary, but assume categoricals are handled)
X = data.select('feature_00_rolling_top_quantile')
y = data['responder_6']

# Convert to DMatrix for XGBoost
dtrain = xgb.DMatrix(X, label=y)

# Define parameters for XGBoost regressor
params = {
    'objective':  'reg:squarederror',
    'colsample_bytree': 0.8,
    'learning_rate': 0.1,
    'max_depth': 6
}

# Train the XGBoost model
xgb_model = xgb.train(params, dtrain, num_boost_round=100)

print('XGBoost model trained successfully.')